# Maze Crawler Miner Hybrid Agent

Experimental Kaggle notebook that keeps the jump-BFS survival core, restores conservative worker timing, and adds one gated miner for mine-economy matchups.


## 1. Algorithm Summary

This notebook starts a separate miner-hybrid family after replay analysis showed that strong opponents can win late games with transformed mines. The key ideas are:

1. Keep jump-preferred factory BFS, remembered walls, mirrored wall inference, and one active replacement scout.
2. Build the first scout before economy logic so early wall discovery remains intact.
3. Build at most one miner only when a visible mining node is close, reachable, and the factory has a safe scroll gap.
4. Transform immediately when the miner reaches a visible mining node.
5. Keep one conservative worker as a backup route opener, using Worker V2-style energy timing.
6. Disable second-scout experiments so miner value can be measured cleanly.
7. Promote this line only if mine economy improves late tiebreaks without creating new scroll deaths.

Worker/scout variants still lose badly to opponents that establish mines. This candidate tests the smallest useful mine economy: one miner, one transform, and strict factory-safety gates.


## 2. Setup

The simulation cells need Kaggle Environments with the `crawl` environment available. Kaggle usually has the package ready, but this cell installs or upgrades it if needed. The next cell centralizes tunable run settings and the replay helper used by both simulations.


In [ ]:
%%capture
!pip install -q "kaggle-environments>=1.29.0"


In [ ]:
from kaggle_environments import make

RANDOM_SEED = 42
RENDER_WIDTH = 800
RENDER_HEIGHT = 800
RUN_SIMULATIONS = True
RUN_BATCH_EVALUATION = False
EVAL_SEEDS = 20
USE_FACTORY_DANGER_GATE = False


def run_simulation(agent_fn, label: str):
    """Run one match against a random opponent and render the replay."""
    if not RUN_SIMULATIONS:
        print(f"{label}: simulation skipped")
        return None

    env = make("crawl", configuration={"randomSeed": RANDOM_SEED}, debug=True)
    env.run([agent_fn, "random"])

    print(label)
    for player_id, step in enumerate(env.steps[-1]):
        print(
            f"Player {player_id}: reward={step.reward}, "
            f"status={step.status}"
        )

    return env.render(
        mode="ipython",
        width=RENDER_WIDTH,
        height=RENDER_HEIGHT,
    )


print("kaggle_environments ready")


## 3. Agent Helpers

These helpers normalize Kaggle objects and dictionaries, cache visible wall information across turns, and mirror known walls across the symmetric board. Unknown cells remain optimistic, which keeps the starter compact while giving BFS more memory than a single observation.


In [ ]:
from collections import deque
from typing import Any

FACTORY = 0
SCOUT = 1
WORKER = 2
MINER = 3

DIRS = {
    "NORTH": (0, 1),
    "SOUTH": (0, -1),
    "EAST": (1, 0),
    "WEST": (-1, 0),
}
MOVE_ORDER = ("NORTH", "EAST", "WEST", "SOUTH")
WALL_BIT = {
    "NORTH": 1,
    "EAST": 2,
    "SOUTH": 4,
    "WEST": 8,
}
MIRROR_WALL = [0] * 16
for value in range(16):
    mirrored = value & (WALL_BIT["NORTH"] | WALL_BIT["SOUTH"])
    if value & WALL_BIT["EAST"]:
        mirrored |= WALL_BIT["WEST"]
    if value & WALL_BIT["WEST"]:
        mirrored |= WALL_BIT["EAST"]
    MIRROR_WALL[value] = mirrored

_wall_memory: dict[tuple[int, int], int] = {}


def get_value(obj: Any, key: str, default: Any = None) -> Any:
    """Return a field from either a dict-like or object-like value.

    Args:
        obj: Observation/config object from Kaggle or a plain dict.
        key: Field name to read.
        default: Fallback when the field is missing.

    Returns:
        The requested value or `default`.
    """
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)


def update_wall_memory(obs: Any, config: Any) -> None:
    """Cache visible walls and mirrored walls for later pathfinding."""
    walls = get_value(obs, "walls", [])
    south_bound = get_value(obs, "southBound", 0)
    width = get_value(config, "width", 0)

    for index, wall_value in enumerate(walls):
        if wall_value == -1:
            continue

        row = south_bound + index // width
        col = index % width
        wall_bits = int(wall_value)
        _wall_memory[(col, row)] = wall_bits

        mirror_col = width - 1 - col
        _wall_memory.setdefault((mirror_col, row), MIRROR_WALL[wall_bits])

    if len(_wall_memory) > 2500:
        cutoff = south_bound - 5
        old_cells = [cell for cell in _wall_memory if cell[1] < cutoff]
        for cell in old_cells:
            del _wall_memory[cell]


def wall_at(obs: Any, config: Any, col: int, row: int) -> int:
    """Return the known wall bitfield at a cell.

    Args:
        obs: Current Maze Crawler observation.
        config: Environment configuration.
        col: Cell column.
        row: Cell row.

    Returns:
        Wall bitfield. Undiscovered cells are treated as open in this starter.
    """
    if (col, row) in _wall_memory:
        return _wall_memory[(col, row)]

    walls = get_value(obs, "walls", [])
    south_bound = get_value(obs, "southBound", 0)
    width = get_value(config, "width", 0)
    index = (row - south_bound) * width + col

    if 0 <= index < len(walls) and walls[index] != -1:
        return int(walls[index])
    return 0


def in_bounds(obs: Any, config: Any, col: int, row: int) -> bool:
    """Return True when a cell is inside the currently playable board."""
    width = get_value(config, "width", 0)
    south_bound = get_value(obs, "southBound", 0)
    north_bound = get_value(obs, "northBound", south_bound + 50)
    return 0 <= col < width and south_bound <= row <= north_bound


def has_road(
    obs: Any,
    config: Any,
    col: int,
    row: int,
    direction: str,
) -> bool:
    """Return True when no known wall blocks a direction from a cell."""
    next_col, next_row = neighbor(col, row, direction)
    if not in_bounds(obs, config, next_col, next_row):
        return False
    return not (wall_at(obs, config, col, row) & WALL_BIT[direction])


def can_jump(
    obs: Any,
    config: Any,
    col: int,
    row: int,
    direction: str,
) -> bool:
    """Return True when a factory jump can land in a plausible cell."""
    delta_col, delta_row = DIRS[direction]
    next_col = col + 2 * delta_col
    next_row = row + 2 * delta_row
    if not in_bounds(obs, config, next_col, next_row):
        return False
    return wall_at(obs, config, next_col, next_row) != 15


def neighbor(col: int, row: int, direction: str) -> tuple[int, int]:
    """Return the adjacent cell in `direction`."""
    delta_col, delta_row = DIRS[direction]
    return col + delta_col, row + delta_row


def jump_neighbor(col: int, row: int, direction: str) -> tuple[int, int]:
    """Return the factory jump destination in `direction`."""
    delta_col, delta_row = DIRS[direction]
    return col + 2 * delta_col, row + 2 * delta_row


def my_robots(obs: Any) -> dict[str, list[int]]:
    """Return robots controlled by the current player."""
    player = get_value(obs, "player", 0)
    robots = get_value(obs, "robots", {}) or {}
    return {
        uid: data
        for uid, data in robots.items()
        if len(data) > 4 and data[4] == player
    }


def my_factory(obs: Any) -> tuple[str | None, list[int] | None]:
    """Return our factory id and data, if visible."""
    for uid, data in my_robots(obs).items():
        if data[0] == FACTORY:
            return uid, data
    return None, None


def occupied_positions(robots: dict[str, list[int]]) -> set[tuple[int, int]]:
    """Return currently occupied cells for our robots."""
    return {(data[1], data[2]) for data in robots.values()}


def parse_cell(key: str) -> tuple[int, int]:
    """Parse a `col,row` map key into integer coordinates."""
    col, row = key.split(",")
    return int(col), int(row)


## 4. Factory Jump-Preferred BFS

The factory now uses a lightweight version of the stronger notebook's main idea: cache wall knowledge, plan toward a higher row with BFS, and explore jump moves before normal walks when the jump cooldown is ready. If the factory is close to the scroll boundary, it switches to emergency escape rules instead of idling in a local dead end.


In [ ]:
def bfs_first_step(
    start: tuple[int, int],
    goals: list[tuple[int, int]],
    obs: Any,
    config: Any,
    depth: int = 20,
    occupied: set[tuple[int, int]] | None = None,
    avoid_occupied: bool = True,
) -> str | None:
    """Return the first walking step to any goal using known open roads."""
    if not goals:
        return None

    occupied = occupied or set()
    goal_set = set(goals)
    if start in goal_set:
        return "IDLE"

    queue = deque([(start, None, 0)])
    seen = {start}

    while queue:
        (col, row), first_step, distance = queue.popleft()
        if (col, row) in goal_set and distance > 0:
            return first_step
        if distance >= depth:
            continue

        for direction in MOVE_ORDER:
            if not has_road(obs, config, col, row, direction):
                continue
            next_cell = neighbor(col, row, direction)
            if next_cell in seen:
                continue
            if avoid_occupied and next_cell in occupied and next_cell != start:
                continue
            seen.add(next_cell)
            queue.append((next_cell, first_step or direction, distance + 1))

    return None


def bfs_distance(
    start: tuple[int, int],
    goals: list[tuple[int, int]],
    obs: Any,
    config: Any,
    depth: int = 20,
    occupied: set[tuple[int, int]] | None = None,
    avoid_occupied: bool = True,
) -> int | None:
    """Return the walking distance to the nearest reachable goal."""
    if not goals:
        return None

    occupied = occupied or set()
    goal_set = set(goals)
    if start in goal_set:
        return 0

    queue = deque([(start, 0)])
    seen = {start}

    while queue:
        (col, row), distance = queue.popleft()
        if distance >= depth:
            continue

        for direction in MOVE_ORDER:
            if not has_road(obs, config, col, row, direction):
                continue
            next_cell = neighbor(col, row, direction)
            if next_cell in seen:
                continue
            if avoid_occupied and next_cell in occupied and next_cell != start:
                continue
            if next_cell in goal_set:
                return distance + 1
            seen.add(next_cell)
            queue.append((next_cell, distance + 1))

    return None


def bfs_jump_preferred(
    start: tuple[int, int],
    goals: list[tuple[int, int]],
    jump_cd: int,
    obs: Any,
    config: Any,
    depth: int = 20,
    occupied: set[tuple[int, int]] | None = None,
) -> str | None:
    """Return a factory step, searching jumps before walks when possible."""
    if not goals:
        return None

    occupied = occupied or set()
    goal_set = set(goals)
    queue = deque([(start, None, 0, max(0, jump_cd))])
    seen = {(start[0], start[1], jump_cd <= 0)}

    while queue:
        (col, row), first_step, distance, current_cd = queue.popleft()
        if (col, row) in goal_set and distance > 0:
            return first_step
        if distance >= depth:
            continue

        if current_cd <= 0:
            for direction in MOVE_ORDER:
                if not can_jump(obs, config, col, row, direction):
                    continue
                next_cell = jump_neighbor(col, row, direction)
                if next_cell in occupied and next_cell != start:
                    continue
                key = (next_cell[0], next_cell[1], False)
                if key in seen:
                    continue
                seen.add(key)
                queue.append((
                    next_cell,
                    first_step or f"JUMP_{direction}",
                    distance + 1,
                    20,
                ))

        for direction in MOVE_ORDER:
            if not has_road(obs, config, col, row, direction):
                continue
            next_cell = neighbor(col, row, direction)
            if next_cell in occupied and next_cell != start:
                continue
            next_cd = max(0, current_cd - 1)
            key = (next_cell[0], next_cell[1], next_cd <= 0)
            if key in seen:
                continue
            seen.add(key)
            queue.append((
                next_cell,
                first_step or direction,
                distance + 1,
                next_cd,
            ))

    return None


def factory_path_action(
    col: int,
    row: int,
    move_cd: int,
    jump_cd: int,
    obs: Any,
    config: Any,
    occupied: set[tuple[int, int]] | None = None,
) -> str:
    """Choose a survival-focused factory action with BFS fallbacks."""
    occupied = occupied or set()
    south_bound = get_value(obs, "southBound", 0)
    north_bound = get_value(obs, "northBound", row + 50)
    width = get_value(config, "width", 0)
    factory_gap = row - south_bound

    if move_cd > 1:
        return "IDLE"

    if factory_gap <= 2 and jump_cd == 0:
        for direction in ("NORTH", "EAST", "WEST"):
            landing = jump_neighbor(col, row, direction)
            if can_jump(obs, config, col, row, direction):
                if landing not in occupied:
                    return f"JUMP_{direction}"

    target_row = min(north_bound, row + 20)
    goals = [(target_col, target_row) for target_col in range(width)]
    step = bfs_jump_preferred(
        (col, row),
        goals,
        jump_cd,
        obs,
        config,
        depth=22,
        occupied=occupied,
    )

    if step is None:
        closer_row = min(north_bound, row + 6)
        closer_goals = [(target_col, closer_row) for target_col in range(width)]
        step = bfs_first_step(
            (col, row),
            closer_goals,
            obs,
            config,
            depth=12,
            occupied=occupied,
            avoid_occupied=False,
        )

    if step is None:
        for direction in ("NORTH", "EAST", "WEST"):
            if has_road(obs, config, col, row, direction):
                next_cell = neighbor(col, row, direction)
                if next_cell not in occupied:
                    return direction

    if step is None and factory_gap <= 3:
        if has_road(obs, config, col, row, "SOUTH"):
            next_cell = neighbor(col, row, "SOUTH")
            if next_cell not in occupied:
                return "SOUTH"

    if step is None and factory_gap <= 3 and jump_cd == 0:
        for direction in ("EAST", "WEST", "SOUTH"):
            landing = jump_neighbor(col, row, direction)
            if can_jump(obs, config, col, row, direction):
                if landing not in occupied:
                    return f"JUMP_{direction}"

    return step or "IDLE"


def agent_v1(obs: Any, config: Any) -> dict[str, str]:
    """Factory-only baseline for a quick survival simulation."""
    update_wall_memory(obs, config)
    actions = {}
    robots = my_robots(obs)
    occupied = occupied_positions(robots)

    for uid, data in robots.items():
        robot_type, col, row = data[0], data[1], data[2]
        move_cd = data[5] if len(data) > 5 else 0
        jump_cd = data[6] if len(data) > 6 else 0
        if robot_type == FACTORY:
            occupied_without_self = occupied - {(col, row)}
            actions[uid] = factory_path_action(
                col,
                row,
                move_cd,
                jump_cd,
                obs,
                config,
                occupied_without_self,
            )
    return actions


## 5. Agent V1 Simulation

Run the factory-only baseline against a random opponent and render the replay. This is the quick sanity check that `crawl` is installed and the basic action dictionary is valid.


In [ ]:
run_simulation(agent_v1, "Agent V1: factory-only baseline")


## 6. Scout, Worker, And Miner Support

The support policy keeps one active scout for vision, adds one miner only near reachable mining nodes, and keeps one conservative worker for route opening. The factory builds the first scout first, then considers miner economy before worker utility.


In [ ]:
SCOUT_RETURN_ENERGY = 60
SCOUT_MINE_TARGET_MAX_DISTANCE = 10
SCOUT_MINE_MIN_STORED_ENERGY = 50
DEFAULT_SCOUT_COST = 50
DEFAULT_WORKER_COST = 200
DEFAULT_MINER_COST = 300
DEFAULT_WALL_REMOVE_COST = 100
DEFAULT_TRANSFORM_COST = 100
MAX_ACTIVE_SCOUTS = 1
MAX_ACTIVE_WORKERS = 1
MAX_ACTIVE_MINERS = 1
WORKER_BUILD_GAP = 8
WORKER_FRONT_OFFSET = 2
WORKER_MIN_FACTORY_ENERGY = 650
MINER_BUILD_GAP = 8
MINER_MIN_FACTORY_ENERGY = 750
MINER_MAX_NODE_DISTANCE = 12
MINE_COLLECT_MIN_GAP = 8
MINE_TARGET_MAX_DISTANCE = 8
MINE_MIN_STORED_ENERGY = 50
USE_FACTORY_DANGER_GATE = False
FACTORY_DANGER_GAP = 4
_last_south_bound: int | None = None
_mining_node_memory: set[tuple[int, int]] = set()


def reset_game_state_if_needed(obs: Any) -> None:
    """Reset persistent state when Kaggle starts a new episode."""
    global _last_south_bound

    south_bound = get_value(obs, "southBound", 0)
    if _last_south_bound is not None and south_bound < _last_south_bound:
        _wall_memory.clear()
        _mining_node_memory.clear()

    _last_south_bound = south_bound


def update_mining_node_memory(obs: Any) -> None:
    """Remember visible mining nodes until they fall behind the scroll."""
    south_bound = get_value(obs, "southBound", 0)
    for key in (get_value(obs, "miningNodes", {}) or {}):
        _mining_node_memory.add(parse_cell(key))

    stale_nodes = [
        node for node in _mining_node_memory if node[1] < south_bound
    ]
    for node in stale_nodes:
        _mining_node_memory.discard(node)


def scout_action(
    col: int,
    row: int,
    energy: int,
    factory_pos: tuple[int, int],
    obs: Any,
    config: Any,
    occupied: set[tuple[int, int]] | None = None,
) -> str:
    """Collect visible crystals and ferry high energy back to the factory."""
    occupied = occupied or set()
    factory_col, factory_row = factory_pos
    if energy >= SCOUT_RETURN_ENERGY:
        for direction in MOVE_ORDER:
            is_adjacent = neighbor(col, row, direction) == (
                factory_col,
                factory_row,
            )
            if is_adjacent and has_road(obs, config, col, row, direction):
                return f"TRANSFER_{direction}"

        step = bfs_first_step(
            (col, row),
            [factory_pos],
            obs,
            config,
            depth=16,
            occupied=occupied,
            avoid_occupied=False,
        )
        return step or "IDLE"

    mine_targets = []
    south_bound = get_value(obs, "southBound", 0)
    for mine, stored_energy in owned_mine_targets(obs):
        mine_gap = mine[1] - south_bound
        distance = abs(mine[0] - col) + abs(mine[1] - row)
        if (
            mine_gap > MINE_COLLECT_MIN_GAP
            and distance <= SCOUT_MINE_TARGET_MAX_DISTANCE
            and stored_energy >= SCOUT_MINE_MIN_STORED_ENERGY
        ):
            route_distance = bfs_distance(
                (col, row),
                [mine],
                obs,
                config,
                depth=SCOUT_MINE_TARGET_MAX_DISTANCE + 2,
                occupied=occupied,
            )
            if route_distance is not None:
                mine_targets.append((route_distance, -stored_energy, mine))

    if mine_targets:
        mine_targets.sort()
        targets = [mine_targets[0][2]]
    else:
        crystals = [
            parse_cell(key)
            for key in (get_value(obs, "crystals", {}) or {})
        ]
        if crystals:
            targets = sorted(
                crystals,
                key=lambda cell: abs(cell[0] - col) + abs(cell[1] - row),
            )[:3]
        else:
            targets = [(factory_col, factory_row + 8)]

    step = bfs_first_step(
        (col, row),
        targets,
        obs,
        config,
        depth=16,
        occupied=occupied,
    )
    if step and step != "IDLE":
        return step

    for direction in ("NORTH", "EAST", "WEST"):
        if has_road(obs, config, col, row, direction):
            next_cell = neighbor(col, row, direction)
            if next_cell not in occupied:
                return direction
    return "IDLE"


def worker_action(
    col: int,
    row: int,
    energy: int,
    factory_pos: tuple[int, int],
    obs: Any,
    config: Any,
    occupied: set[tuple[int, int]] | None = None,
) -> str:
    """Open a route ahead of the factory without adding broad economy logic."""
    occupied = occupied or set()
    remove_cost = get_value(
        config,
        "wallRemoveCost",
        DEFAULT_WALL_REMOVE_COST,
    )
    if energy >= remove_cost and not has_road(obs, config, col, row, "NORTH"):
        return "REMOVE_NORTH"

    factory_col, factory_row = factory_pos
    target = (factory_col, factory_row + WORKER_FRONT_OFFSET)
    step = bfs_first_step(
        (col, row),
        [target],
        obs,
        config,
        depth=10,
        occupied=occupied,
    )
    if step and step != "IDLE":
        return step

    for direction in ("NORTH", "EAST", "WEST"):
        if has_road(obs, config, col, row, direction):
            next_cell = neighbor(col, row, direction)
            if next_cell not in occupied:
                return direction
    return "IDLE"


def visible_mining_nodes(obs: Any) -> list[tuple[int, int]]:
    """Return currently visible mining nodes as coordinate tuples."""
    return [
        parse_cell(key)
        for key in (get_value(obs, "miningNodes", {}) or {})
    ]


def remembered_mining_nodes(obs: Any) -> list[tuple[int, int]]:
    """Return visible and remembered mining nodes still above the scroll."""
    south_bound = get_value(obs, "southBound", 0)
    nodes = set(_mining_node_memory) | set(visible_mining_nodes(obs))
    return sorted(node for node in nodes if node[1] >= south_bound)


def close_reachable_mining_nodes(
    start: tuple[int, int],
    obs: Any,
    config: Any,
    occupied: set[tuple[int, int]] | None = None,
) -> list[tuple[int, int]]:
    """Return remembered mining nodes that are plausible miner targets."""
    occupied = occupied or set()
    candidates = []
    for node in remembered_mining_nodes(obs):
        manhattan = abs(node[0] - start[0]) + abs(node[1] - start[1])
        if manhattan > MINER_MAX_NODE_DISTANCE:
            continue
        distance = bfs_distance(
            start,
            [node],
            obs,
            config,
            depth=MINER_MAX_NODE_DISTANCE,
            occupied=occupied,
        )
        if distance is not None:
            candidates.append((distance, node))

    return [node for _, node in sorted(candidates)]


def miner_action(
    col: int,
    row: int,
    energy: int,
    obs: Any,
    config: Any,
    occupied: set[tuple[int, int]] | None = None,
) -> str:
    """Transform on mining nodes or route a miner toward the closest node."""
    occupied = occupied or set()
    transform_cost = get_value(
        config,
        "transformCost",
        DEFAULT_TRANSFORM_COST,
    )
    if (col, row) in set(visible_mining_nodes(obs)) and energy >= transform_cost:
        return "TRANSFORM"

    targets = close_reachable_mining_nodes(
        (col, row),
        obs,
        config,
        occupied=occupied,
    )
    if targets:
        step = bfs_first_step(
            (col, row),
            targets[:3],
            obs,
            config,
            depth=MINER_MAX_NODE_DISTANCE + 4,
            occupied=occupied,
        )
        if step and step != "IDLE":
            return step

    for direction in ("NORTH", "EAST", "WEST"):
        if has_road(obs, config, col, row, direction):
            next_cell = neighbor(col, row, direction)
            if next_cell not in occupied:
                return direction
    return "IDLE"


def owned_mine_targets(obs: Any) -> list[tuple[tuple[int, int], int]]:
    """Return owned mines as `(cell, stored_energy)` pairs."""
    player = get_value(obs, "player", 0)
    targets = []
    for key, data in (get_value(obs, "mines", {}) or {}).items():
        if len(data) > 2 and data[2] == player:
            targets.append((parse_cell(key), int(data[0])))
    return targets


def factory_mine_collect_action(
    col: int,
    row: int,
    move_cd: int,
    obs: Any,
    config: Any,
    occupied: set[tuple[int, int]] | None = None,
) -> str | None:
    """Move the factory toward a nearby safe owned mine."""
    occupied = occupied or set()
    south_bound = get_value(obs, "southBound", 0)
    factory_gap = row - south_bound
    if factory_gap <= MINE_COLLECT_MIN_GAP:
        return None

    candidates = []
    for mine, stored_energy in owned_mine_targets(obs):
        mine_gap = mine[1] - south_bound
        if mine_gap <= MINE_COLLECT_MIN_GAP:
            continue
        distance = abs(mine[0] - col) + abs(mine[1] - row)
        if distance > MINE_TARGET_MAX_DISTANCE:
            continue
        if mine != (col, row) and stored_energy < MINE_MIN_STORED_ENERGY:
            continue
        if mine[1] < row - 1:
            continue

        route_distance = bfs_distance(
            (col, row),
            [mine],
            obs,
            config,
            depth=MINE_TARGET_MAX_DISTANCE + 2,
            occupied=occupied,
        )
        if route_distance is not None:
            candidates.append((route_distance, -stored_energy, mine))

    if not candidates:
        return None

    candidates.sort()
    target = candidates[0][2]
    if target == (col, row):
        return "IDLE"
    if move_cd > 1:
        return None

    step = bfs_first_step(
        (col, row),
        [target],
        obs,
        config,
        depth=MINE_TARGET_MAX_DISTANCE + 2,
        occupied=occupied,
    )
    if step and step != "IDLE":
        return step
    return None


def can_spawn_scout(
    col: int,
    row: int,
    energy: int,
    build_cd: int,
    scout_count: int,
    obs: Any,
    config: Any,
    occupied: set[tuple[int, int]],
) -> bool:
    """Return True when the factory can safely spawn the first scout."""
    scout_cost = get_value(config, "scoutCost", DEFAULT_SCOUT_COST)
    spawn = neighbor(col, row, "NORTH")
    south_bound = get_value(obs, "southBound", 0)
    factory_gap = row - south_bound
    danger_gate_ok = (
        not USE_FACTORY_DANGER_GATE
        or factory_gap > FACTORY_DANGER_GAP
    )
    return (
        scout_count == 0
        and danger_gate_ok
        and build_cd == 0
        and energy >= scout_cost
        and spawn not in occupied
        and has_road(obs, config, col, row, "NORTH")
    )




def can_spawn_miner(
    col: int,
    row: int,
    energy: int,
    build_cd: int,
    obs: Any,
    config: Any,
    occupied: set[tuple[int, int]],
) -> bool:
    """Return True when one miner can target a nearby visible node."""
    miner_cost = get_value(config, "minerCost", DEFAULT_MINER_COST)
    spawn = neighbor(col, row, "NORTH")
    south_bound = get_value(obs, "southBound", 0)
    factory_gap = row - south_bound
    if not (
        factory_gap > MINER_BUILD_GAP
        and build_cd == 0
        and energy >= max(miner_cost, MINER_MIN_FACTORY_ENERGY)
        and spawn not in occupied
        and has_road(obs, config, col, row, "NORTH")
    ):
        return False

    return bool(
        close_reachable_mining_nodes(
            spawn,
            obs,
            config,
            occupied=occupied,
        )
    )


def can_spawn_worker(
    col: int,
    row: int,
    energy: int,
    build_cd: int,
    scout_count: int,
    obs: Any,
    config: Any,
    occupied: set[tuple[int, int]],
) -> bool:
    """Return True when the worker experiment can safely spend energy."""
    worker_cost = get_value(config, "workerCost", DEFAULT_WORKER_COST)
    spawn = neighbor(col, row, "NORTH")
    south_bound = get_value(obs, "southBound", 0)
    factory_gap = row - south_bound
    return (
        scout_count > 0
        and factory_gap > WORKER_BUILD_GAP
        and build_cd == 0
        and energy >= max(worker_cost, WORKER_MIN_FACTORY_ENERGY)
        and spawn not in occupied
        and has_road(obs, config, col, row, "NORTH")
    )


def agent_v2(obs: Any, config: Any) -> dict[str, str]:
    """Jump-BFS reference with one scout, one miner, and one worker."""
    reset_game_state_if_needed(obs)
    update_wall_memory(obs, config)
    update_mining_node_memory(obs)
    actions = {}
    robots = my_robots(obs)
    _, factory_data = my_factory(obs)
    occupied = occupied_positions(robots)
    scout_count = sum(1 for data in robots.values() if data[0] == SCOUT)
    worker_count = sum(1 for data in robots.values() if data[0] == WORKER)
    miner_count = sum(1 for data in robots.values() if data[0] == MINER)

    for uid, data in robots.items():
        robot_type, col, row, energy = data[0], data[1], data[2], data[3]
        move_cd = data[5] if len(data) > 5 else 0
        jump_cd = data[6] if len(data) > 6 else 0
        build_cd = data[7] if len(data) > 7 else 0
        occupied_without_self = occupied - {(col, row)}

        if robot_type == FACTORY:
            if scout_count == 0 and can_spawn_scout(
                col,
                row,
                energy,
                build_cd,
                scout_count,
                obs,
                config,
                occupied_without_self,
            ):
                actions[uid] = "BUILD_SCOUT"
                scout_count += 1
            elif miner_count < MAX_ACTIVE_MINERS and can_spawn_miner(
                col,
                row,
                energy,
                build_cd,
                obs,
                config,
                occupied_without_self,
            ):
                actions[uid] = "BUILD_MINER"
                miner_count += 1
            elif worker_count < MAX_ACTIVE_WORKERS and can_spawn_worker(
                col,
                row,
                energy,
                build_cd,
                scout_count,
                obs,
                config,
                occupied_without_self,
            ):
                actions[uid] = "BUILD_WORKER"
                worker_count += 1
            else:
                mine_action = factory_mine_collect_action(
                    col,
                    row,
                    move_cd,
                    obs,
                    config,
                    occupied_without_self,
                )
                if mine_action is not None:
                    actions[uid] = mine_action
                else:
                    actions[uid] = factory_path_action(
                        col,
                        row,
                        move_cd,
                        jump_cd,
                        obs,
                        config,
                        occupied_without_self,
                    )
        elif robot_type == SCOUT and factory_data is not None:
            factory_pos = (factory_data[1], factory_data[2])
            actions[uid] = scout_action(
                col,
                row,
                energy,
                factory_pos,
                obs,
                config,
                occupied_without_self,
            )
        elif robot_type == WORKER and factory_data is not None:
            factory_pos = (factory_data[1], factory_data[2])
            actions[uid] = worker_action(
                col,
                row,
                energy,
                factory_pos,
                obs,
                config,
                occupied_without_self,
            )
        elif robot_type == MINER:
            actions[uid] = miner_action(
                col,
                row,
                energy,
                obs,
                config,
                occupied_without_self,
            )

    return actions


## 7. Agent V2 Miner Hybrid Simulation


In [ ]:
_wall_memory.clear()
_last_south_bound = None
run_simulation(agent_v2, "Agent V3: remembered-node miner timing test")


## 8. Generate Submission Files

Kaggle imports `agent` from `main.py`. This cell writes the final agent source to `main.py`; the next cell mirrors that exact source to `submission.py` for tooling that expects the alternate filename. Keep generated files out of git.


In [ ]:
%%writefile main.py
from collections import deque
from typing import Any

FACTORY = 0
SCOUT = 1
WORKER = 2
MINER = 3

DIRS = {
    "NORTH": (0, 1),
    "SOUTH": (0, -1),
    "EAST": (1, 0),
    "WEST": (-1, 0),
}
MOVE_ORDER = ("NORTH", "EAST", "WEST", "SOUTH")
WALL_BIT = {
    "NORTH": 1,
    "EAST": 2,
    "SOUTH": 4,
    "WEST": 8,
}
MIRROR_WALL = [0] * 16
for value in range(16):
    mirrored = value & (WALL_BIT["NORTH"] | WALL_BIT["SOUTH"])
    if value & WALL_BIT["EAST"]:
        mirrored |= WALL_BIT["WEST"]
    if value & WALL_BIT["WEST"]:
        mirrored |= WALL_BIT["EAST"]
    MIRROR_WALL[value] = mirrored

_wall_memory: dict[tuple[int, int], int] = {}


def get_value(obj: Any, key: str, default: Any = None) -> Any:
    """Return a field from either a dict-like or object-like value.

    Args:
        obj: Observation/config object from Kaggle or a plain dict.
        key: Field name to read.
        default: Fallback when the field is missing.

    Returns:
        The requested value or `default`.
    """
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)


def update_wall_memory(obs: Any, config: Any) -> None:
    """Cache visible walls and mirrored walls for later pathfinding."""
    walls = get_value(obs, "walls", [])
    south_bound = get_value(obs, "southBound", 0)
    width = get_value(config, "width", 0)

    for index, wall_value in enumerate(walls):
        if wall_value == -1:
            continue

        row = south_bound + index // width
        col = index % width
        wall_bits = int(wall_value)
        _wall_memory[(col, row)] = wall_bits

        mirror_col = width - 1 - col
        _wall_memory.setdefault((mirror_col, row), MIRROR_WALL[wall_bits])

    if len(_wall_memory) > 2500:
        cutoff = south_bound - 5
        old_cells = [cell for cell in _wall_memory if cell[1] < cutoff]
        for cell in old_cells:
            del _wall_memory[cell]


def wall_at(obs: Any, config: Any, col: int, row: int) -> int:
    """Return the known wall bitfield at a cell.

    Args:
        obs: Current Maze Crawler observation.
        config: Environment configuration.
        col: Cell column.
        row: Cell row.

    Returns:
        Wall bitfield. Undiscovered cells are treated as open in this starter.
    """
    if (col, row) in _wall_memory:
        return _wall_memory[(col, row)]

    walls = get_value(obs, "walls", [])
    south_bound = get_value(obs, "southBound", 0)
    width = get_value(config, "width", 0)
    index = (row - south_bound) * width + col

    if 0 <= index < len(walls) and walls[index] != -1:
        return int(walls[index])
    return 0


def in_bounds(obs: Any, config: Any, col: int, row: int) -> bool:
    """Return True when a cell is inside the currently playable board."""
    width = get_value(config, "width", 0)
    south_bound = get_value(obs, "southBound", 0)
    north_bound = get_value(obs, "northBound", south_bound + 50)
    return 0 <= col < width and south_bound <= row <= north_bound


def has_road(
    obs: Any,
    config: Any,
    col: int,
    row: int,
    direction: str,
) -> bool:
    """Return True when no known wall blocks a direction from a cell."""
    next_col, next_row = neighbor(col, row, direction)
    if not in_bounds(obs, config, next_col, next_row):
        return False
    return not (wall_at(obs, config, col, row) & WALL_BIT[direction])


def can_jump(
    obs: Any,
    config: Any,
    col: int,
    row: int,
    direction: str,
) -> bool:
    """Return True when a factory jump can land in a plausible cell."""
    delta_col, delta_row = DIRS[direction]
    next_col = col + 2 * delta_col
    next_row = row + 2 * delta_row
    if not in_bounds(obs, config, next_col, next_row):
        return False
    return wall_at(obs, config, next_col, next_row) != 15


def neighbor(col: int, row: int, direction: str) -> tuple[int, int]:
    """Return the adjacent cell in `direction`."""
    delta_col, delta_row = DIRS[direction]
    return col + delta_col, row + delta_row


def jump_neighbor(col: int, row: int, direction: str) -> tuple[int, int]:
    """Return the factory jump destination in `direction`."""
    delta_col, delta_row = DIRS[direction]
    return col + 2 * delta_col, row + 2 * delta_row


def my_robots(obs: Any) -> dict[str, list[int]]:
    """Return robots controlled by the current player."""
    player = get_value(obs, "player", 0)
    robots = get_value(obs, "robots", {}) or {}
    return {
        uid: data
        for uid, data in robots.items()
        if len(data) > 4 and data[4] == player
    }


def my_factory(obs: Any) -> tuple[str | None, list[int] | None]:
    """Return our factory id and data, if visible."""
    for uid, data in my_robots(obs).items():
        if data[0] == FACTORY:
            return uid, data
    return None, None


def occupied_positions(robots: dict[str, list[int]]) -> set[tuple[int, int]]:
    """Return currently occupied cells for our robots."""
    return {(data[1], data[2]) for data in robots.values()}


def parse_cell(key: str) -> tuple[int, int]:
    """Parse a `col,row` map key into integer coordinates."""
    col, row = key.split(",")
    return int(col), int(row)

def bfs_first_step(
    start: tuple[int, int],
    goals: list[tuple[int, int]],
    obs: Any,
    config: Any,
    depth: int = 20,
    occupied: set[tuple[int, int]] | None = None,
    avoid_occupied: bool = True,
) -> str | None:
    """Return the first walking step to any goal using known open roads."""
    if not goals:
        return None

    occupied = occupied or set()
    goal_set = set(goals)
    if start in goal_set:
        return "IDLE"

    queue = deque([(start, None, 0)])
    seen = {start}

    while queue:
        (col, row), first_step, distance = queue.popleft()
        if (col, row) in goal_set and distance > 0:
            return first_step
        if distance >= depth:
            continue

        for direction in MOVE_ORDER:
            if not has_road(obs, config, col, row, direction):
                continue
            next_cell = neighbor(col, row, direction)
            if next_cell in seen:
                continue
            if avoid_occupied and next_cell in occupied and next_cell != start:
                continue
            seen.add(next_cell)
            queue.append((next_cell, first_step or direction, distance + 1))

    return None


def bfs_distance(
    start: tuple[int, int],
    goals: list[tuple[int, int]],
    obs: Any,
    config: Any,
    depth: int = 20,
    occupied: set[tuple[int, int]] | None = None,
    avoid_occupied: bool = True,
) -> int | None:
    """Return the walking distance to the nearest reachable goal."""
    if not goals:
        return None

    occupied = occupied or set()
    goal_set = set(goals)
    if start in goal_set:
        return 0

    queue = deque([(start, 0)])
    seen = {start}

    while queue:
        (col, row), distance = queue.popleft()
        if distance >= depth:
            continue

        for direction in MOVE_ORDER:
            if not has_road(obs, config, col, row, direction):
                continue
            next_cell = neighbor(col, row, direction)
            if next_cell in seen:
                continue
            if avoid_occupied and next_cell in occupied and next_cell != start:
                continue
            if next_cell in goal_set:
                return distance + 1
            seen.add(next_cell)
            queue.append((next_cell, distance + 1))

    return None


def bfs_jump_preferred(
    start: tuple[int, int],
    goals: list[tuple[int, int]],
    jump_cd: int,
    obs: Any,
    config: Any,
    depth: int = 20,
    occupied: set[tuple[int, int]] | None = None,
) -> str | None:
    """Return a factory step, searching jumps before walks when possible."""
    if not goals:
        return None

    occupied = occupied or set()
    goal_set = set(goals)
    queue = deque([(start, None, 0, max(0, jump_cd))])
    seen = {(start[0], start[1], jump_cd <= 0)}

    while queue:
        (col, row), first_step, distance, current_cd = queue.popleft()
        if (col, row) in goal_set and distance > 0:
            return first_step
        if distance >= depth:
            continue

        if current_cd <= 0:
            for direction in MOVE_ORDER:
                if not can_jump(obs, config, col, row, direction):
                    continue
                next_cell = jump_neighbor(col, row, direction)
                if next_cell in occupied and next_cell != start:
                    continue
                key = (next_cell[0], next_cell[1], False)
                if key in seen:
                    continue
                seen.add(key)
                queue.append((
                    next_cell,
                    first_step or f"JUMP_{direction}",
                    distance + 1,
                    20,
                ))

        for direction in MOVE_ORDER:
            if not has_road(obs, config, col, row, direction):
                continue
            next_cell = neighbor(col, row, direction)
            if next_cell in occupied and next_cell != start:
                continue
            next_cd = max(0, current_cd - 1)
            key = (next_cell[0], next_cell[1], next_cd <= 0)
            if key in seen:
                continue
            seen.add(key)
            queue.append((
                next_cell,
                first_step or direction,
                distance + 1,
                next_cd,
            ))

    return None


def factory_path_action(
    col: int,
    row: int,
    move_cd: int,
    jump_cd: int,
    obs: Any,
    config: Any,
    occupied: set[tuple[int, int]] | None = None,
) -> str:
    """Choose a survival-focused factory action with BFS fallbacks."""
    occupied = occupied or set()
    south_bound = get_value(obs, "southBound", 0)
    north_bound = get_value(obs, "northBound", row + 50)
    width = get_value(config, "width", 0)
    factory_gap = row - south_bound

    if move_cd > 1:
        return "IDLE"

    if factory_gap <= 2 and jump_cd == 0:
        for direction in ("NORTH", "EAST", "WEST"):
            landing = jump_neighbor(col, row, direction)
            if can_jump(obs, config, col, row, direction):
                if landing not in occupied:
                    return f"JUMP_{direction}"

    target_row = min(north_bound, row + 20)
    goals = [(target_col, target_row) for target_col in range(width)]
    step = bfs_jump_preferred(
        (col, row),
        goals,
        jump_cd,
        obs,
        config,
        depth=22,
        occupied=occupied,
    )

    if step is None:
        closer_row = min(north_bound, row + 6)
        closer_goals = [(target_col, closer_row) for target_col in range(width)]
        step = bfs_first_step(
            (col, row),
            closer_goals,
            obs,
            config,
            depth=12,
            occupied=occupied,
            avoid_occupied=False,
        )

    if step is None:
        for direction in ("NORTH", "EAST", "WEST"):
            if has_road(obs, config, col, row, direction):
                next_cell = neighbor(col, row, direction)
                if next_cell not in occupied:
                    return direction

    if step is None and factory_gap <= 3:
        if has_road(obs, config, col, row, "SOUTH"):
            next_cell = neighbor(col, row, "SOUTH")
            if next_cell not in occupied:
                return "SOUTH"

    if step is None and factory_gap <= 3 and jump_cd == 0:
        for direction in ("EAST", "WEST", "SOUTH"):
            landing = jump_neighbor(col, row, direction)
            if can_jump(obs, config, col, row, direction):
                if landing not in occupied:
                    return f"JUMP_{direction}"

    return step or "IDLE"


def agent_v1(obs: Any, config: Any) -> dict[str, str]:
    """Factory-only baseline for a quick survival simulation."""
    update_wall_memory(obs, config)
    actions = {}
    robots = my_robots(obs)
    occupied = occupied_positions(robots)

    for uid, data in robots.items():
        robot_type, col, row = data[0], data[1], data[2]
        move_cd = data[5] if len(data) > 5 else 0
        jump_cd = data[6] if len(data) > 6 else 0
        if robot_type == FACTORY:
            occupied_without_self = occupied - {(col, row)}
            actions[uid] = factory_path_action(
                col,
                row,
                move_cd,
                jump_cd,
                obs,
                config,
                occupied_without_self,
            )
    return actions

SCOUT_RETURN_ENERGY = 60
SCOUT_MINE_TARGET_MAX_DISTANCE = 10
SCOUT_MINE_MIN_STORED_ENERGY = 50
DEFAULT_SCOUT_COST = 50
DEFAULT_WORKER_COST = 200
DEFAULT_MINER_COST = 300
DEFAULT_WALL_REMOVE_COST = 100
DEFAULT_TRANSFORM_COST = 100
MAX_ACTIVE_SCOUTS = 1
MAX_ACTIVE_WORKERS = 1
MAX_ACTIVE_MINERS = 1
WORKER_BUILD_GAP = 8
WORKER_FRONT_OFFSET = 2
WORKER_MIN_FACTORY_ENERGY = 650
MINER_BUILD_GAP = 8
MINER_MIN_FACTORY_ENERGY = 750
MINER_MAX_NODE_DISTANCE = 12
MINE_COLLECT_MIN_GAP = 8
MINE_TARGET_MAX_DISTANCE = 8
MINE_MIN_STORED_ENERGY = 50
USE_FACTORY_DANGER_GATE = False
FACTORY_DANGER_GAP = 4
_last_south_bound: int | None = None
_mining_node_memory: set[tuple[int, int]] = set()


def reset_game_state_if_needed(obs: Any) -> None:
    """Reset persistent state when Kaggle starts a new episode."""
    global _last_south_bound

    south_bound = get_value(obs, "southBound", 0)
    if _last_south_bound is not None and south_bound < _last_south_bound:
        _wall_memory.clear()
        _mining_node_memory.clear()

    _last_south_bound = south_bound


def update_mining_node_memory(obs: Any) -> None:
    """Remember visible mining nodes until they fall behind the scroll."""
    south_bound = get_value(obs, "southBound", 0)
    for key in (get_value(obs, "miningNodes", {}) or {}):
        _mining_node_memory.add(parse_cell(key))

    stale_nodes = [
        node for node in _mining_node_memory if node[1] < south_bound
    ]
    for node in stale_nodes:
        _mining_node_memory.discard(node)


def scout_action(
    col: int,
    row: int,
    energy: int,
    factory_pos: tuple[int, int],
    obs: Any,
    config: Any,
    occupied: set[tuple[int, int]] | None = None,
) -> str:
    """Collect visible crystals and ferry high energy back to the factory."""
    occupied = occupied or set()
    factory_col, factory_row = factory_pos
    if energy >= SCOUT_RETURN_ENERGY:
        for direction in MOVE_ORDER:
            is_adjacent = neighbor(col, row, direction) == (
                factory_col,
                factory_row,
            )
            if is_adjacent and has_road(obs, config, col, row, direction):
                return f"TRANSFER_{direction}"

        step = bfs_first_step(
            (col, row),
            [factory_pos],
            obs,
            config,
            depth=16,
            occupied=occupied,
            avoid_occupied=False,
        )
        return step or "IDLE"

    mine_targets = []
    south_bound = get_value(obs, "southBound", 0)
    for mine, stored_energy in owned_mine_targets(obs):
        mine_gap = mine[1] - south_bound
        distance = abs(mine[0] - col) + abs(mine[1] - row)
        if (
            mine_gap > MINE_COLLECT_MIN_GAP
            and distance <= SCOUT_MINE_TARGET_MAX_DISTANCE
            and stored_energy >= SCOUT_MINE_MIN_STORED_ENERGY
        ):
            route_distance = bfs_distance(
                (col, row),
                [mine],
                obs,
                config,
                depth=SCOUT_MINE_TARGET_MAX_DISTANCE + 2,
                occupied=occupied,
            )
            if route_distance is not None:
                mine_targets.append((route_distance, -stored_energy, mine))

    if mine_targets:
        mine_targets.sort()
        targets = [mine_targets[0][2]]
    else:
        crystals = [
            parse_cell(key)
            for key in (get_value(obs, "crystals", {}) or {})
        ]
        if crystals:
            targets = sorted(
                crystals,
                key=lambda cell: abs(cell[0] - col) + abs(cell[1] - row),
            )[:3]
        else:
            targets = [(factory_col, factory_row + 8)]

    step = bfs_first_step(
        (col, row),
        targets,
        obs,
        config,
        depth=16,
        occupied=occupied,
    )
    if step and step != "IDLE":
        return step

    for direction in ("NORTH", "EAST", "WEST"):
        if has_road(obs, config, col, row, direction):
            next_cell = neighbor(col, row, direction)
            if next_cell not in occupied:
                return direction
    return "IDLE"


def worker_action(
    col: int,
    row: int,
    energy: int,
    factory_pos: tuple[int, int],
    obs: Any,
    config: Any,
    occupied: set[tuple[int, int]] | None = None,
) -> str:
    """Open a route ahead of the factory without adding broad economy logic."""
    occupied = occupied or set()
    remove_cost = get_value(
        config,
        "wallRemoveCost",
        DEFAULT_WALL_REMOVE_COST,
    )
    if energy >= remove_cost and not has_road(obs, config, col, row, "NORTH"):
        return "REMOVE_NORTH"

    factory_col, factory_row = factory_pos
    target = (factory_col, factory_row + WORKER_FRONT_OFFSET)
    step = bfs_first_step(
        (col, row),
        [target],
        obs,
        config,
        depth=10,
        occupied=occupied,
    )
    if step and step != "IDLE":
        return step

    for direction in ("NORTH", "EAST", "WEST"):
        if has_road(obs, config, col, row, direction):
            next_cell = neighbor(col, row, direction)
            if next_cell not in occupied:
                return direction
    return "IDLE"


def visible_mining_nodes(obs: Any) -> list[tuple[int, int]]:
    """Return currently visible mining nodes as coordinate tuples."""
    return [
        parse_cell(key)
        for key in (get_value(obs, "miningNodes", {}) or {})
    ]


def remembered_mining_nodes(obs: Any) -> list[tuple[int, int]]:
    """Return visible and remembered mining nodes still above the scroll."""
    south_bound = get_value(obs, "southBound", 0)
    nodes = set(_mining_node_memory) | set(visible_mining_nodes(obs))
    return sorted(node for node in nodes if node[1] >= south_bound)


def close_reachable_mining_nodes(
    start: tuple[int, int],
    obs: Any,
    config: Any,
    occupied: set[tuple[int, int]] | None = None,
) -> list[tuple[int, int]]:
    """Return remembered mining nodes that are plausible miner targets."""
    occupied = occupied or set()
    candidates = []
    for node in remembered_mining_nodes(obs):
        manhattan = abs(node[0] - start[0]) + abs(node[1] - start[1])
        if manhattan > MINER_MAX_NODE_DISTANCE:
            continue
        distance = bfs_distance(
            start,
            [node],
            obs,
            config,
            depth=MINER_MAX_NODE_DISTANCE,
            occupied=occupied,
        )
        if distance is not None:
            candidates.append((distance, node))

    return [node for _, node in sorted(candidates)]


def miner_action(
    col: int,
    row: int,
    energy: int,
    obs: Any,
    config: Any,
    occupied: set[tuple[int, int]] | None = None,
) -> str:
    """Transform on mining nodes or route a miner toward the closest node."""
    occupied = occupied or set()
    transform_cost = get_value(
        config,
        "transformCost",
        DEFAULT_TRANSFORM_COST,
    )
    if (col, row) in set(visible_mining_nodes(obs)) and energy >= transform_cost:
        return "TRANSFORM"

    targets = close_reachable_mining_nodes(
        (col, row),
        obs,
        config,
        occupied=occupied,
    )
    if targets:
        step = bfs_first_step(
            (col, row),
            targets[:3],
            obs,
            config,
            depth=MINER_MAX_NODE_DISTANCE + 4,
            occupied=occupied,
        )
        if step and step != "IDLE":
            return step

    for direction in ("NORTH", "EAST", "WEST"):
        if has_road(obs, config, col, row, direction):
            next_cell = neighbor(col, row, direction)
            if next_cell not in occupied:
                return direction
    return "IDLE"


def owned_mine_targets(obs: Any) -> list[tuple[tuple[int, int], int]]:
    """Return owned mines as `(cell, stored_energy)` pairs."""
    player = get_value(obs, "player", 0)
    targets = []
    for key, data in (get_value(obs, "mines", {}) or {}).items():
        if len(data) > 2 and data[2] == player:
            targets.append((parse_cell(key), int(data[0])))
    return targets


def factory_mine_collect_action(
    col: int,
    row: int,
    move_cd: int,
    obs: Any,
    config: Any,
    occupied: set[tuple[int, int]] | None = None,
) -> str | None:
    """Move the factory toward a nearby safe owned mine."""
    occupied = occupied or set()
    south_bound = get_value(obs, "southBound", 0)
    factory_gap = row - south_bound
    if factory_gap <= MINE_COLLECT_MIN_GAP:
        return None

    candidates = []
    for mine, stored_energy in owned_mine_targets(obs):
        mine_gap = mine[1] - south_bound
        if mine_gap <= MINE_COLLECT_MIN_GAP:
            continue
        distance = abs(mine[0] - col) + abs(mine[1] - row)
        if distance > MINE_TARGET_MAX_DISTANCE:
            continue
        if mine != (col, row) and stored_energy < MINE_MIN_STORED_ENERGY:
            continue
        if mine[1] < row - 1:
            continue

        route_distance = bfs_distance(
            (col, row),
            [mine],
            obs,
            config,
            depth=MINE_TARGET_MAX_DISTANCE + 2,
            occupied=occupied,
        )
        if route_distance is not None:
            candidates.append((route_distance, -stored_energy, mine))

    if not candidates:
        return None

    candidates.sort()
    target = candidates[0][2]
    if target == (col, row):
        return "IDLE"
    if move_cd > 1:
        return None

    step = bfs_first_step(
        (col, row),
        [target],
        obs,
        config,
        depth=MINE_TARGET_MAX_DISTANCE + 2,
        occupied=occupied,
    )
    if step and step != "IDLE":
        return step
    return None


def can_spawn_scout(
    col: int,
    row: int,
    energy: int,
    build_cd: int,
    scout_count: int,
    obs: Any,
    config: Any,
    occupied: set[tuple[int, int]],
) -> bool:
    """Return True when the factory can safely spawn the first scout."""
    scout_cost = get_value(config, "scoutCost", DEFAULT_SCOUT_COST)
    spawn = neighbor(col, row, "NORTH")
    south_bound = get_value(obs, "southBound", 0)
    factory_gap = row - south_bound
    danger_gate_ok = (
        not USE_FACTORY_DANGER_GATE
        or factory_gap > FACTORY_DANGER_GAP
    )
    return (
        scout_count == 0
        and danger_gate_ok
        and build_cd == 0
        and energy >= scout_cost
        and spawn not in occupied
        and has_road(obs, config, col, row, "NORTH")
    )




def can_spawn_miner(
    col: int,
    row: int,
    energy: int,
    build_cd: int,
    obs: Any,
    config: Any,
    occupied: set[tuple[int, int]],
) -> bool:
    """Return True when one miner can target a nearby visible node."""
    miner_cost = get_value(config, "minerCost", DEFAULT_MINER_COST)
    spawn = neighbor(col, row, "NORTH")
    south_bound = get_value(obs, "southBound", 0)
    factory_gap = row - south_bound
    if not (
        factory_gap > MINER_BUILD_GAP
        and build_cd == 0
        and energy >= max(miner_cost, MINER_MIN_FACTORY_ENERGY)
        and spawn not in occupied
        and has_road(obs, config, col, row, "NORTH")
    ):
        return False

    return bool(
        close_reachable_mining_nodes(
            spawn,
            obs,
            config,
            occupied=occupied,
        )
    )


def can_spawn_worker(
    col: int,
    row: int,
    energy: int,
    build_cd: int,
    scout_count: int,
    obs: Any,
    config: Any,
    occupied: set[tuple[int, int]],
) -> bool:
    """Return True when the worker experiment can safely spend energy."""
    worker_cost = get_value(config, "workerCost", DEFAULT_WORKER_COST)
    spawn = neighbor(col, row, "NORTH")
    south_bound = get_value(obs, "southBound", 0)
    factory_gap = row - south_bound
    return (
        scout_count > 0
        and factory_gap > WORKER_BUILD_GAP
        and build_cd == 0
        and energy >= max(worker_cost, WORKER_MIN_FACTORY_ENERGY)
        and spawn not in occupied
        and has_road(obs, config, col, row, "NORTH")
    )


def agent_v2(obs: Any, config: Any) -> dict[str, str]:
    """Jump-BFS reference with one scout, one miner, and one worker."""
    reset_game_state_if_needed(obs)
    update_wall_memory(obs, config)
    update_mining_node_memory(obs)
    actions = {}
    robots = my_robots(obs)
    _, factory_data = my_factory(obs)
    occupied = occupied_positions(robots)
    scout_count = sum(1 for data in robots.values() if data[0] == SCOUT)
    worker_count = sum(1 for data in robots.values() if data[0] == WORKER)
    miner_count = sum(1 for data in robots.values() if data[0] == MINER)

    for uid, data in robots.items():
        robot_type, col, row, energy = data[0], data[1], data[2], data[3]
        move_cd = data[5] if len(data) > 5 else 0
        jump_cd = data[6] if len(data) > 6 else 0
        build_cd = data[7] if len(data) > 7 else 0
        occupied_without_self = occupied - {(col, row)}

        if robot_type == FACTORY:
            if scout_count == 0 and can_spawn_scout(
                col,
                row,
                energy,
                build_cd,
                scout_count,
                obs,
                config,
                occupied_without_self,
            ):
                actions[uid] = "BUILD_SCOUT"
                scout_count += 1
            elif miner_count < MAX_ACTIVE_MINERS and can_spawn_miner(
                col,
                row,
                energy,
                build_cd,
                obs,
                config,
                occupied_without_self,
            ):
                actions[uid] = "BUILD_MINER"
                miner_count += 1
            elif worker_count < MAX_ACTIVE_WORKERS and can_spawn_worker(
                col,
                row,
                energy,
                build_cd,
                scout_count,
                obs,
                config,
                occupied_without_self,
            ):
                actions[uid] = "BUILD_WORKER"
                worker_count += 1
            else:
                mine_action = factory_mine_collect_action(
                    col,
                    row,
                    move_cd,
                    obs,
                    config,
                    occupied_without_self,
                )
                if mine_action is not None:
                    actions[uid] = mine_action
                else:
                    actions[uid] = factory_path_action(
                        col,
                        row,
                        move_cd,
                        jump_cd,
                        obs,
                        config,
                        occupied_without_self,
                    )
        elif robot_type == SCOUT and factory_data is not None:
            factory_pos = (factory_data[1], factory_data[2])
            actions[uid] = scout_action(
                col,
                row,
                energy,
                factory_pos,
                obs,
                config,
                occupied_without_self,
            )
        elif robot_type == WORKER and factory_data is not None:
            factory_pos = (factory_data[1], factory_data[2])
            actions[uid] = worker_action(
                col,
                row,
                energy,
                factory_pos,
                obs,
                config,
                occupied_without_self,
            )
        elif robot_type == MINER:
            actions[uid] = miner_action(
                col,
                row,
                energy,
                obs,
                config,
                occupied_without_self,
            )

    return actions

def compute_actions(obs: Any, config: Any) -> dict[str, str]:
    """Return the production starter action dictionary."""
    return agent_v2(obs, config)


def idle_actions(obs: Any) -> dict[str, str]:
    """Return safe idle actions for visible owned robots."""
    return {uid: "IDLE" for uid in my_robots(obs)}


def agent(obs: Any, config: Any) -> dict[str, str]:
    """Kaggle-facing entry point."""
    try:
        return compute_actions(obs, config)
    except Exception:
        return idle_actions(obs)


def act(obs: Any, config: Any) -> dict[str, str]:
    """Alias used by some local runners."""
    return agent(obs, config)


__all__ = ["agent", "act", "compute_actions"]


## 9. Verify Generated Files

The generated files should compile and stay byte-for-byte identical. Keep this check close to the write cell so stale submission artifacts are obvious.


In [ ]:
from pathlib import Path
import py_compile

main_path = Path("main.py")
submission_path = Path("submission.py")
submission_path.write_text(
    main_path.read_text(encoding="utf-8"),
    encoding="utf-8",
)

py_compile.compile(str(main_path), doraise=True)
py_compile.compile(str(submission_path), doraise=True)
assert main_path.read_text(encoding="utf-8") == submission_path.read_text(
    encoding="utf-8"
)

print("main.py: OK")
print("submission.py: OK")
print("submission.py sync: OK")


## 10. Batch Evaluation

Optional multi-seed evaluation for local signal. Keep `RUN_BATCH_EVALUATION = False` for normal Kaggle submission runs, then enable it when comparing policy variants.


In [ ]:
def summarize_match(agent_fn, seed: int) -> dict[str, int | str]:
    """Run one random-opponent match and summarize key signals."""
    global _last_south_bound
    _wall_memory.clear()
    _last_south_bound = None

    env = make("crawl", configuration={"randomSeed": seed}, debug=True)
    env.run([agent_fn, "random"])

    counts: dict[str, int] = {}
    min_gap = 999
    worker_seen = False
    worker_alive_final = False
    worker_final_energy = 0
    miner_seen = False
    miner_alive_final = False
    miner_final_energy = 0

    for step in env.steps:
        state = step[0]
        for action in (state.action or {}).values():
            counts[action] = counts.get(action, 0) + 1

        obs = state.observation
        robots = getattr(obs, "robots", {}) or {}
        player = getattr(obs, "player", 0)
        south_bound = getattr(obs, "southBound", 0)
        current_worker_energy = 0
        current_worker_alive = False
        current_miner_energy = 0
        current_miner_alive = False

        for data in robots.values():
            if len(data) <= 4 or data[4] != player:
                continue
            if data[0] == FACTORY:
                min_gap = min(min_gap, data[2] - south_bound)
            elif data[0] == WORKER:
                worker_seen = True
                current_worker_alive = True
                current_worker_energy += data[3]
            elif data[0] == MINER:
                miner_seen = True
                current_miner_alive = True
                current_miner_energy += data[3]

        worker_alive_final = current_worker_alive
        worker_final_energy = current_worker_energy
        miner_alive_final = current_miner_alive
        miner_final_energy = current_miner_energy

    final = env.steps[-1][0]
    return {
        "seed": seed,
        "reward": final.reward,
        "status": final.status,
        "steps": len(env.steps) - 1,
        "build_scout": counts.get("BUILD_SCOUT", 0),
        "build_worker": counts.get("BUILD_WORKER", 0),
        "build_miner": counts.get("BUILD_MINER", 0),
        "transform": counts.get("TRANSFORM", 0),
        "removes": sum(
            value for key, value in counts.items() if key.startswith("REMOVE_")
        ),
        "worker_seen": int(worker_seen),
        "worker_alive_final": int(worker_alive_final),
        "worker_final_energy": worker_final_energy,
        "miner_seen": miner_seen,
        "miner_alive_final": miner_alive_final,
        "miner_final_energy": miner_final_energy,
        "transfers": sum(
            value for key, value in counts.items() if key.startswith("TRANSFER_")
        ),
        "jumps": sum(
            value for key, value in counts.items() if key.startswith("JUMP_")
        ),
        "idle": counts.get("IDLE", 0),
        "min_factory_gap": min_gap,
    }


if RUN_SIMULATIONS and RUN_BATCH_EVALUATION:
    eval_rows = []
    for seed in range(EVAL_SEEDS):
        eval_rows.append({"agent": "v1", **summarize_match(agent_v1, seed)})
        eval_rows.append({"agent": "v2", **summarize_match(agent_v2, seed)})
    for row in eval_rows:
        print(row)


## 11. Submit To The Leaderboard

After running the notebook on Kaggle, use **Save Version** and then **Submit to Competition**. The generated `main.py` is the file Kaggle evaluates.


## 12. Next Improvements

This version keeps V5 scout-assisted mine targeting and lowers the scout return threshold so carried mine energy is transferred back sooner. Useful next checks after Kaggle submission:

- Confirm `BUILD_MINER` and `TRANSFORM` remain nonzero in replay traces.
- Compare `TRANSFER_*` counts against V5.
- Compare final factory energy and scout-on-mine counts against V5.
- Watch for scouts returning too early and losing crystal exploration value.
- If transfers improve but score falls, try `SCOUT_RETURN_ENERGY = 65` as a softer midpoint.
